# Model performance and sensitivity

Native reconstruction comparisons for Figure 5 and Supplementary Figures 3–5. Each setup uses a complete native ensemble and the same observed national mortality series.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/04_model_performance_sensitivity.ipynb).

In [ ]:
from pathlib import Path
import sys

repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs" / "training.example.yaml").is_file())
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from notebooks.utils import daily_index, ensemble, national_daily

from notebooks.utils import ANALYSIS_PERIOD, DEVICE, checkpoint_paths
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor


## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to a value from 0 through 9 to inspect another member. The analysis cells evaluate all ten members in a separate inference workspace.

In [ ]:
inspection_member = 0
groups = {
    "Input components": {
        "SARI and COVID-19 only": "exclude_temperature",
        "Temperature only": "temperature_only",
        "Weather and total SARI": "no_covid",
        "Full hybrid": "default",
    },
    "Baseline": {
        "Static": "baseline_static",
        "Median-EWMA": "default",
        "Trimmed mean": "baseline_trimmed_mean",
        "Double-sided Median-EWMA": "baseline_double_sided",
    },
    "Humidity indicator": {
        "Temperature and dew point": "default",
        "Temperature only": "tas_only",
        "Temperature and wet bulb": "tas_wet_bulb",
    },
    "Calibration": {"Enabled": "default", "Disabled": "no_mean_alignment"},
}
setups = dict.fromkeys(setup for group in groups.values() for setup in group.values())
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Evaluate the native experiment registry

In [ ]:
summaries = dict.fromkeys(setups, national_daily)

estimates_by_setup = {}
for setup, summarize_member in summaries.items():
    paths = checkpoints[setup]
    inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
    inference_model = InferenceModel.from_checkpoint(
        paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
    )
    estimates_by_setup[setup] = ensemble(inference_model, paths, summarize_member)
    del inference_model

predictions = estimates_by_setup
dates = daily_index()
observed = predictions["default"]["Observed"].numpy()
if any(not np.array_equal(observed, values["Observed"].numpy())
       for values in predictions.values()):
    raise ValueError("All setups must use the same observed labels.")

## Reconstruction metrics by period

In [ ]:
week = dates.isocalendar().week.to_numpy()
year = dates.year
masks = {
    "Overall": np.ones(len(dates), dtype=bool),
    "Influenza season": (week >= 40) | (week <= 20),
    "Summer": (week >= 15) & (week <= 38),
    "Pre-pandemic": year <= 2019,
    "Pandemic": (year >= 2020) & (year <= 2022),
    "2023–2024": (year >= 2023) & (year <= 2024),
    "2025 test": year == 2025,
}
rows = []
for setup, values in predictions.items():
    predicted = values["Predicted"].numpy()
    for window, mask in masks.items():
        reference = observed[mask]
        squared_errors = (predicted[mask] - reference) ** 2
        r2 = 1 - squared_errors.sum() / ((reference - reference.mean()) ** 2).sum()
        rows.append((setup, window, np.sqrt(squared_errors.mean()), r2))
metrics = pd.DataFrame(rows, columns=["Setup", "Period", "RMSE", "R²"])
metrics

## Sensitivity panels

In [ ]:
for title, group in groups.items():
    names = {setup: label for label, setup in group.items()}
    subset = metrics[metrics["Setup"].isin(names)].assign(
        Model=lambda frame: frame["Setup"].map(names)
    )
    rmse = subset.pivot(index="Model", columns="Period", values="RMSE")
    rmse = rmse.reindex(index=list(group), columns=list(masks))
    ax = rmse.T.plot.bar(figsize=(12, 4), title=title, ylabel="Daily RMSE")
    ax.legend(frameon=False)
    ax.figure.tight_layout()